# S&P 500 Companies Dataset Cleaning and Validation

## Overview

This section documents the cleaning and validation process applied to the **S&P 500 Companies dataset**, which contains company-level metadata for all firms included in the S&P 500 index.

This dataset provides important contextual information about each company, including:

- ticker symbol
- company name
- sector classification
- industry classification
- headquarters location
- date added to the index
- CIK identifier
- company founding year

Unlike the previous datasets (stock prices and the S&P 500 index), which contain **time-series financial data**, this dataset represents **static company metadata**.  

The purpose of cleaning this dataset is to ensure that it can be reliably integrated with the stock price dataset using the **ticker symbol as the join key**.

The cleaning pipeline focuses on:

- column standardization
- ticker validation
- duplicate detection
- missing value analysis
- structural validation

---

In [1]:
import pandas as pd

# =========================================================
# LOAD S&P 500 COMPANY METADATA
# =========================================================

companies_path = "../../data/raw/sp500_companies.csv"

sp500_companies = pd.read_csv(companies_path)

print("===================================================")
print("RAW S&P500 COMPANIES DATASET LOADED")
print("===================================================\n")

print("Shape:")
print(sp500_companies.shape)

print("\nColumns:")
print(sp500_companies.columns)

print("\nPreview:")
print(sp500_companies.head())

RAW S&P500 COMPANIES DATASET LOADED

Shape:
(503, 8)

Columns:
Index(['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry',
       'Headquarters Location', 'Date added', 'CIK', 'Founded'],
      dtype='object')

Preview:
  Symbol             Security             GICS Sector  \
0    MMM                   3M             Industrials   
1    AOS          A. O. Smith             Industrials   
2    ABT  Abbott Laboratories             Health Care   
3   ABBV               AbbVie             Health Care   
4    ACN            Accenture  Information Technology   

                GICS Sub-Industry    Headquarters Location  Date added  \
0        Industrial Conglomerates    Saint Paul, Minnesota    3/4/1957   
1               Building Products     Milwaukee, Wisconsin   7/26/2017   
2           Health Care Equipment  North Chicago, Illinois    3/4/1957   
3                   Biotechnology  North Chicago, Illinois  12/31/2012   
4  IT Consulting & Other Services          Dublin, Ireland    

# Step 1: Loading the Raw Dataset

The raw S&P 500 companies dataset was loaded from the project’s **raw data directory**.

Initial dataset characteristics:

Shape:

(503, 8)

This confirms that the dataset contains:

- **503 companies**
- **8 variables**

This matches the expected size of the S&P 500 index, which typically contains approximately 500 companies but may include slightly more due to historical inclusion records.

---

In [2]:
print("\n===================================================")
print("DATASET STRUCTURE")
print("===================================================\n")

print(sp500_companies.info())


DATASET STRUCTURE

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Symbol                 503 non-null    object
 1   Security               503 non-null    object
 2   GICS Sector            503 non-null    object
 3   GICS Sub-Industry      503 non-null    object
 4   Headquarters Location  503 non-null    object
 5   Date added             503 non-null    object
 6   CIK                    503 non-null    int64 
 7   Founded                503 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.6+ KB
None


# Step 2: Inspecting Dataset Structure

The dataset structure was inspected using dataframe information functions.

The raw dataset contains the following columns:

Symbol  
Security  
GICS Sector  
GICS Sub-Industry  
Headquarters Location  
Date added  
CIK  
Founded  

Data types identified:

Symbol → object  
Security → object  
GICS Sector → object  
GICS Sub-Industry → object  
Headquarters Location → object  
Date added → object  
CIK → int64  
Founded → object  

The inspection confirmed that all columns contain **503 non-null values**, indicating that the dataset is structurally complete.

---

In [3]:
print("\n===================================================")
print("MISSING VALUE ANALYSIS")
print("===================================================\n")

missing_values = sp500_companies.isna().sum()

print(missing_values)


MISSING VALUE ANALYSIS

Symbol                   0
Security                 0
GICS Sector              0
GICS Sub-Industry        0
Headquarters Location    0
Date added               0
CIK                      0
Founded                  0
dtype: int64


In [4]:
# =========================================================
# STANDARDIZE COLUMN NAMES
# =========================================================

sp500_companies = sp500_companies.rename(columns={
    "Symbol": "Symbol",
    "Security": "Company",
    "GICS Sector": "Sector",
    "GICS Sub-Industry": "Industry",
    "Headquarters Location": "Headquarters",
    "Date added": "Date_Added",
    "CIK": "CIK",
    "Founded": "Founded"
})

print("Columns after renaming:")
print(sp500_companies.columns)

Columns after renaming:
Index(['Symbol', 'Company', 'Sector', 'Industry', 'Headquarters', 'Date_Added',
       'CIK', 'Founded'],
      dtype='object')


# Step 3: Standardizing Column Names

To maintain consistency across the project pipeline, column names were standardized.

The following renaming transformations were applied:

| Original Column | New Column Name |
|----------------|----------------|
| Symbol | Symbol |
| Security | Company |
| GICS Sector | Sector |
| GICS Sub-Industry | Industry |
| Headquarters Location | Headquarters |
| Date added | Date_Added |
| CIK | CIK |
| Founded | Founded |

The purpose of this step is to:

- simplify column naming
- remove spaces
- ensure compatibility with Python scripts and SQL queries
- maintain consistency with other datasets in the project

Final column names after renaming:

Symbol  
Company  
Sector  
Industry  
Headquarters  
Date_Added  
CIK  
Founded  

---

In [5]:
print("\n===================================================")
print("TICKER SYMBOL VALIDATION")
print("===================================================\n")

unique_symbols = sp500_companies["Symbol"].nunique()

print("Unique ticker symbols:", unique_symbols)


TICKER SYMBOL VALIDATION

Unique ticker symbols: 503


In [6]:
print("\nChecking duplicate symbols...")

duplicates = sp500_companies["Symbol"].duplicated().sum()

print("Duplicate symbols:", duplicates)

sp500_companies = sp500_companies.drop_duplicates(subset="Symbol")


Checking duplicate symbols...
Duplicate symbols: 0


# Step 4: Ticker Symbol Validation

The **Symbol column** serves as the **primary key** for joining this dataset with the stock prices dataset.

Therefore, it is essential to verify that:

1. all ticker symbols are unique  
2. no duplicate tickers exist  

Validation results:

Unique ticker symbols:

503

Duplicate symbols:

0

This confirms that every company is uniquely identified by its ticker symbol, ensuring that the dataset can be safely used for dataset integration.


In [7]:
# =========================================================
# DATA TYPE STANDARDIZATION
# =========================================================

sp500_companies["Date_Added"] = pd.to_datetime(
    sp500_companies["Date_Added"],
    errors="coerce"
)

sp500_companies["Founded"] = pd.to_numeric(
    sp500_companies["Founded"],
    errors="coerce"
)

In [8]:
print("\n===================================================")
print("POST-CLEANING VALIDATION")
print("===================================================\n")

print("Shape:", sp500_companies.shape)

print("\nMissing values:")
print(sp500_companies.isna().sum())

print("\nDuplicate Symbols:")
print(sp500_companies["Symbol"].duplicated().sum())


POST-CLEANING VALIDATION

Shape: (503, 8)

Missing values:
Symbol           0
Company          0
Sector           0
Industry         0
Headquarters     0
Date_Added       0
CIK              0
Founded         41
dtype: int64

Duplicate Symbols:
0


In [9]:
sp500_companies_clean = sp500_companies.copy()

sp500_companies_clean.to_csv(
    "../../data/cleaned/sp500_companies_clean.csv",
    index=False
)

print("\nClean companies dataset saved successfully.")


Clean companies dataset saved successfully.


# Step 6: Post-Cleaning Validation

After completing the cleaning steps, a final validation was performed.

### Dataset Shape

Final dataset shape:

(503, 8)

This confirms that no rows were removed during the cleaning process.

---

### Duplicate Symbol Check

Duplicate ticker symbols:

0

This confirms that each company appears exactly once in the dataset.

---

### Missing Value Summary

Only one column contains missing values:

Founded → 41

All other columns are fully populated.

---

# Final Result

After completing the cleaning and validation process, the dataset is confirmed to be:

- structurally consistent
- free of duplicate ticker symbols
- suitable for integration with the stock prices dataset
- complete in all essential fields required for analysis

The dataset now contains clean company metadata for **503 S&P 500 companies**.

---

# Project Status

At this stage, the cleaning phase of the data pipeline is complete.

All three datasets have now been cleaned and validated:

1. **Stock Prices Dataset**
2. **S&P 500 Index Dataset**
3. **S&P 500 Companies Dataset**

Each dataset has been:

- validated for structural integrity
- cleaned and standardized
- checked for duplicates
- checked for missing values
- saved to the **clean data layer**

---

# Next Step: Data Integration

With all datasets cleaned, the next stage of the project will be **dataset integration**, where the cleaned datasets will be combined to create a **master financial dataset**.

The integration will combine:

- stock price data  
- market index data  
- company metadata  

This final dataset will support financial analysis, modeling, and visualization.